In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [ ]:
query = """

SELECT
    reference_period,
    buyer_nfc, seller_nfc,
    currency1, currency2,
    notional_eur
FROM lab_prj_emir_ecb.hermesf_fx
WHERE contract_type = 'FORW'

"""

df = pd.read_sql_query(query, cnxn)

In [ ]:
# One position record per entity-leg (both sides count if both are our entities)
buy = df[df.buyer_nfc == 1].assign(side='buyer')
sell = df[df.seller_nfc == 1].assign(side='seller')
legs = pd.concat([buy, sell], ignore_index=True)

# Foreign-currency direction (EUR = home). Buyer is long currency1, seller short currency1;
# a long EUR leg means a short foreign-currency position.
buyer_long_fx = legs.currency1 != 'EUR'
legs['direction'] = np.where(
    legs.side == 'buyer',
    np.where(buyer_long_fx, 'long', 'short'),
    np.where(buyer_long_fx, 'short', 'long'),
)

# Outstanding notional (EUR) per reference period, split by direction
legs['reference_period'] = pd.to_datetime(legs.reference_period)
ts = (legs.groupby(['reference_period', 'direction'])['notional_eur']
          .sum().unstack(fill_value=0).sort_index())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(ts.index, ts['long'] / 1e9, label='Long FX')
ax.plot(ts.index, ts['short'] / 1e9, label='Short FX')
ax.set_xlabel('Reference period')
ax.set_ylabel('Outstanding notional (EUR bn)')
ax.set_title('Outstanding forward positions of HermesF entities')
ax.legend()
plt.show()